Vamos ahora a tratar las imputaciones registradas con el sistema MES

En las imputaciones es donde esta el detalle real de ejecución (qué máquina, qué tiempos reales, quién y cuándo).

In [494]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [495]:
ruta_imputations = os.path.join("res\\raw", "imputaciones.csv")
print(ruta_imputations)

res\raw\imputaciones.csv


In [496]:
columns_imputations = ["FechaCreacionDte",	"smNodoId",	"smNodoNme",	"smOrdenId",	"smMaterialId",	"smOrdenNme",	"smFaseId",	"smFaseNme",	"CantidadManualOKNbr",	"CantidadManualNOKNbr",	"MinutosRealNbr",	"MinutosPreparacionNbr",	"MinutosProduccionNbr",	"PorcentajeDedicacionNbr",	"ProductivoInd",	"smIncidenciaId",	"smIncidenciaNme",	"smFasePersonalizado1Txt",	"smNodoId2",	"smNodoPersonalizado1Nbr",	"smNodoPersonalizado2Nbr",	"ssEnumFaseEstadoId",	"SmFaseUid",	"TraspasadoInd",	"JsonBodyTxt",	"JsonResponseTxt"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)

C:\Users\jaume\AppData\Local\Temp\ipykernel_21544\1223014052.py:6: DtypeWarning: Columns (0: smFaseId, 1: smIncidenciaId, 2: smIncidenciaNme) have mixed types. Specify dtype option on import or set low_memory=False.
  df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)


In [497]:
display(df_imputations.sample(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseId,smFaseNme,CantidadManualOKNbr,CantidadManualNOKNbr,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,smFasePersonalizado1Txt,smNodoId2,smNodoPersonalizado1Nbr,smNodoPersonalizado2Nbr,ssEnumFaseEstadoId,SmFaseUid,TraspasadoInd,JsonBodyTxt,JsonResponseTxt
160150,2025-04-16 21:15:12.467,CM15,C5 IBARMIA ZVH45,OF2503528,CODXA1X0046,BARRA DE INTERCONEXION BB32231800002,15,MEC04,0.0,0.0,0.0,0.0,0.0,0.0,1,NaN,NaN,NaN,CM15,NaN,NaN,FINALIZADA_OLANET,3A188EFE-3882-40A4-BB51-3DCA0D92F3A9,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[]}"


In [498]:
display(df_imputations.shape)

(196576, 26)

## Limpieza de atributos
Eliminar del dataset aquellos atributos prescindibles. 


In [499]:
def eliminar_columnas(df):
    """
    Elimina de df_imputations las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original).
    """

    cols_sobran = [
        "JsonResponseTxt", "JsonBodyTxt", "TraspasadoInd", "ssEnumFaseEstadoId", 
        "smNodoPersonalizado2Nbr", "smNodoPersonalizado1Nbr", "smNodoId2",
        "smFasePersonalizado1Txt", "PorcentajeDedicacionNbr", "CantidadManualOKNbr",
        "CantidadManualNOKNbr", "smFaseId",
        
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_imputations]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_imputations.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [500]:
df_imputations_limpieza = eliminar_columnas(df_imputations)

Eliminadas 12 columnas. Quedan 14 columnas, 196576 filas.


In [501]:
display(df_imputations_limpieza.sample(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid
30564,2026-04-24 14:12:23.430,TN07,HYUNDAI-KIA SKT21 LMS Torn C.N.C,OF2603551,3128767,Volandera,TCN,0.0,0.0,0.0,1,NaN,NaN,CC0F047B-DAA3-4594-AF73-290D185DD9A6


### Limpieza por intervalo de fechas
Trabajamos dentro del rango entre junio 2025 y antes de agosto 2026.

In [502]:
def filtrar_desde_fecha(df, fecha_desde, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea >= fecha_desde.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro >= {fecha_desde}.")

    limite = pd.to_datetime(fecha_desde)
    df_out = df_out[df_out[columna] >= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (desde {fecha_desde}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [503]:
df_imputations_limpieza = filtrar_desde_fecha(df_imputations_limpieza, "2025-06-01")

Partida: 196576 fases
Conservadas (desde 2025-06-01): 141329 (descartadas 55247)


In [504]:
def filtrar_hasta_fecha(df, fecha_hasta, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea =< fecha_hasta.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro =< {fecha_hasta}.")

    limite = pd.to_datetime(fecha_hasta)
    df_out = df_out[df_out[columna] <= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (hasta {fecha_hasta}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [505]:
df_imputations_limpieza = filtrar_hasta_fecha(df_imputations_limpieza, "2026-08-01")

Partida: 141329 fases
Conservadas (hasta 2026-08-01): 140485 (descartadas 844)


Voy a guarda la tabla de las imputaciones en `res`

In [506]:
import pyarrow as pa

#df_imputations_limpieza["smFaseId"] = df_imputations_limpieza["smFaseId"].astype(str)

df_imputations_limpieza.to_parquet("res/procesed/df_imputations_limpieza.parquet")

## Filtrar imputaciones por maquinas productivas
Todas las imputaciones que no pertenezcan a la selección de maquinas productivas (`grupo_maquinas`) debo hacerlas desaparecer ya que son imputaciones a no emplear para mi proyecto. 

In [507]:
# cargo el df de maquinas y grupos
maquina_grupo = pd.read_parquet("res//procesed/maquina_grupo.parquet")
display(maquina_grupo.sample(1))
print(f"Total de máquinas: {len(maquina_grupo)}")
print(f"Grupos de máquinas: {len(maquina_grupo["CodigoGrupo"].unique())}")

#print(len(maquina_grupo["CodigoMaquina"].unique()))
#print(len(maquina_grupo["CodigoMaquina"]))

,IDMaquina,CodigoMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
55,733e8446-eeed-4b61-9125-7d873719dca1,CM14,CV DOOSAN DNM-750L,01a3c08e-a022-49c4-9c19-0f1fffd63df7,CV,Centres de Mecanitzat Vertical COURES


Total de máquinas: 32
Grupos de máquinas: 9


> Para comprender mejor la siguiente celda del código, quiero remarcar que para `maquinas_validas[CodigoMaquina]` tiene su codigo homologo en `df_imputations_limpieza[smNodoId]`. Esto permite relacionar facilmente ambas tamblas mediante el código de la máquina. 

In [508]:
maquinas_validas = maquina_grupo["CodigoMaquina"].unique()
print(f"Imputaciones antes: {len(df_imputations_limpieza)}")
df_imput_productivas = df_imputations_limpieza[
    df_imputations_limpieza["smNodoId"].isin(maquinas_validas)
]
print(f"Imputaciones después: {len(df_imput_productivas)}")
print(f"Eliminadas (imputaciones en máquinas no productivas): {len(df_imputations_limpieza) - len(df_imput_productivas)}")

Imputaciones antes: 140485
Imputaciones después: 59574
Eliminadas (imputaciones en máquinas no productivas): 80911


In [509]:
fuera = round(((len(df_imputations_limpieza)-len(df_imput_productivas))/len(df_imputations_limpieza))*100, 2)
print(f"Desaparecen el {fuera}% de las imputaciones porque pertenecen a máquinas fuera de la població de estudio")

Desaparecen el 57.59% de las imputaciones porque pertenecen a máquinas fuera de la població de estudio


## Resolver fases imputadas en N nodos
Interesa averiguar si se ha dado una misma fase en distintas maquinas, por ello debo verificar si todas las imputaciones de una misma fase se han hecho en la misma máquina.   
Para cada imputación podemos saber en que nodo se ha realizado, un nodo puede ser una maquina o un centro de trabajo.

In [510]:
maquinas_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()
print(f"Total de fases productivas: {len(df_imput_productivas.groupby('SmFaseUid'))}")
print("")
print("Entre las fases productivas, el número de máquinas distintas por fase es:")
print(maquinas_por_fase.value_counts())

Total de fases productivas: 19149

Entre las fases productivas, el número de máquinas distintas por fase es:
smNodoId
1    18250
2      860
3       37
4        2
Name: count, dtype: int64


Averiguar que maquinas son las que presentan imputaciones de una misma fase en distintos nodos es innecesario, lo que realmente me interesa es ver si este cambio de nodo traspasa entre distintos grupos de máquinas:

In [511]:
# recuento de nodos por fase (sobre imputaciones productivas)
nodos_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()

fases_mono = nodos_por_fase[nodos_por_fase == 1].index
fases_multi = nodos_por_fase[nodos_por_fase > 1].index

# --- 1. Peso en número de fases ---
total_fases = len(nodos_por_fase)
n_multi = len(fases_multi)
print(f"Fases totales:            {total_fases}")
print(f"Fases mono-nodo:          {len(fases_mono)} ({len(fases_mono)/total_fases:.1%})")
print(f"Fases multi-nodo:         {n_multi} ({n_multi/total_fases:.1%})")

# --- 2. Peso en tiempo real (¿son marginales también en volumen de trabajo?) ---
tiempo_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]["MinutosRealNbr"].sum()
tiempo_total = df_imput_productivas["MinutosRealNbr"].sum()
print(f"\nTiempo real en fases multi-nodo: {tiempo_multi/tiempo_total:.1%} del total")

Fases totales:            19149
Fases mono-nodo:          18250 (95.3%)
Fases multi-nodo:         899 (4.7%)

Tiempo real en fases multi-nodo: 15.6% del total


Hay pocas fases en multi-nodo (4,6%), pero con peso desproporcionado en el tiempo total de todas las imputaciones (14,5%). No son ruido marginal, sino fases más "pesadas" que la media.      
Entonces el 4,6% de las fases acumula el 14,5% del tiempo real. Eso significa que una fase multi-nodo tienen más tiempo real que una fase normal. Hablando con el equipo de producción tiene sentido lógico, son fases largas o complejas, y precisamente por eso se repartieron entre varias máquinas o sesiones.

> Voy a inspeccionarlas.

¿En que máquinas aparecen las fases multi nodo?

In [512]:
# ¿en qué máquinas aparecen las fases multi-nodo?
imput_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]

t = imput_multi.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()
fraccion_dominante = t.groupby("SmFaseUid").max() / t.groupby("SmFaseUid").sum()
print(f"Total de fases donde la maquina dominante concentra >80% del tiempo: {(fraccion_dominante > 0.8).mean():.1%}")

Total de fases donde la maquina dominante concentra >80% del tiempo: 33.5%


En el 35,6% de las fases multi-nodo la máquina dominante concentra más del 80% del tiempo, es decir, en el 63,2% de las imputaciones en `df_imput_productivas`, el tiempo está genuinamente repartido, la "maquina dominante" no domina tanto, convive con otras máquinas con peso real.  
  
> Voy a trabajar con ese 36,8% de fases donde es facilmente identificable la maquina dominante, para las 63,2% restantes voy a prescindir, obviando las mismas para este proyecto. 

In [513]:
# tiempo por (fase, nodo)
tiempo_fase_nodo = df_imput_productivas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()

# fracción de la dominante en cada fase
dominante = tiempo_fase_nodo.groupby("SmFaseUid").max()
total = tiempo_fase_nodo.groupby("SmFaseUid").sum()
fraccion_dominante = dominante / total

# fases válidas: la dominante concentra >80% (esto incluye automáticamente las mono-nodo, que son 100%)
fases_validas = fraccion_dominante[fraccion_dominante > 0.8].index

print(f"Fases totales antes:     {fraccion_dominante.shape[0]}")
print(f"Fases válidas (dominante >80%): {len(fases_validas)}")
print(f"Fases descartadas (repartidas): {fraccion_dominante.shape[0] - len(fases_validas)}")

Fases totales antes:     19149
Fases válidas (dominante >80%): 18548
Fases descartadas (repartidas): 601


#### Revisar incidencias
Hay imputaciones en maquinas que corresponden a incidencias, inspecciono cuantas son y planteo descartarlas, ya que estas no tocan tiempo productivo.

In [514]:
# Revisar incidencias:
df_incidencias = df_imput_productivas[df_imput_productivas["smIncidenciaId"].notnull()]  
print(df_incidencias.shape)

(4, 14)


In [515]:
pre_eliminar_incidencias = df_imput_productivas.copy()
print(f"Fases contando con incidencias: {len(pre_eliminar_incidencias)}")
df_imput_productivas = df_imput_productivas[df_imput_productivas["smIncidenciaId"].isnull()]
print(f"Fases sin contar con incidencias: {len(df_imput_productivas)}")
print(f"Total de incidencias eliminadas: {len(pre_eliminar_incidencias) - len(df_imput_productivas)}")

Fases contando con incidencias: 59574
Fases sin contar con incidencias: 59570
Total de incidencias eliminadas: 4


Entonces partimos de 140.485 imputaciones, el recorte a operaciones productivas elimina el 52,54%, dejando 66.668 imputaciones repartidas en 21830 fases productivas, de las cuales nos quedamos con 21186 donde podemos identificar una maquina dominante como la responsable de la realización del trabajo en la fase. 

## Tabla final imputaciones: una instancia por cada fase
Finalmente voy a construir la tabla, con una fila por fase válida, con su grupo con la máquina dominante, y sus tiempos reales agregados.

In [516]:
display(df_imput_productivas.head(0))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid


In [517]:
# 1. me quedo solo con las imputaciones de las fases válidas
imput_validas = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_validas)]

# 2. máquina dominante de cada fase (el nodo con más tiempo real)
    # Agrupa por la pareja fase + nodo, y suma el tiempo real de cada combinación.
tiempo_fase_nodo = imput_validas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum().reset_index()
    # Encontrar, por fase, la fila del tiempo máximo:
idx_dom = tiempo_fase_nodo.groupby("SmFaseUid")["MinutosRealNbr"].idxmax()
    # Recuperar esas filas y quedarse con fase + nodo
dominante = tiempo_fase_nodo.loc[idx_dom, ["SmFaseUid", "smNodoId"]].rename(columns={"smNodoId": "MaquinaDominante"})

# 3. tiempos agregados por fase (suma de TODAS sus imputaciones)
tiempos = imput_validas.groupby("SmFaseUid").agg(
    smOrdenId=("smOrdenId", "first"),
    smMaterialId=("smMaterialId", "first"),
    smFaseNme=("smFaseNme", "first"),
    smOrdenNme=("smOrdenNme", "first"),
    MinutosReal_total=("MinutosRealNbr", "sum"),
    MinutosPreparacion_total=("MinutosPreparacionNbr", "sum"),
    MinutosProduccion_total=("MinutosProduccionNbr", "sum"),
    n_imputaciones=("MinutosRealNbr", "size"),
).reset_index()

# 4. uno máquina dominante + tiempos en la tabla final (1 fila por fase)
fase_agregada = dominante.merge(tiempos, on="SmFaseUid", how="left")

print(f"Fases en la tabla final: {len(fase_agregada)}")
display(fase_agregada.sample(5))

Fases en la tabla final: 18548


,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,n_imputaciones
908,0CADEA01-2B97-43CD-88AA-C162BB4D7F72,CM16,OF2605727,PE1XNCX0371,PE01,Elektrodenaufnahme F704501046762,305.33334,59.00000,246.33334,5
5754,50473B69-9D96-411F-824A-940BFA19D7AC,CM08,OF2602234,BALX60X0051,CV-BAL,Pinole 125x40 DS013148(6159320T101),83.96667,83.96667,0.00000,4
10009,8B3E06FB-2757-4462-819A-2CE773B607C7,TN04,OF2508748,BR1XA3X0037,TPG-CNY,ARM B1PN54CA0567,87.80000,87.80000,0.00000,2
10383,901EFDBC-D46A-43D2-8470-059DC66D4A94,TP02,OF2507070,PE1XA2X0638,MTP,ALLONGE COUDÉE 103019971,34.88333,0.00000,0.00000,2
15646,D7DB181F-9798-4620-B7F9-1D0075128A6C,FL01,OF2508278,CFFXA1X0111,FLX,Shunt curt C-50 1159D324921L,236.26668,40.86667,195.40001,5


### Revisar el correcto traspaso de tiempos en las agrupaciones:
Los minutos totales se calcula de la siguiente manera:
`MinutosReal_total = MinutosPreparacion_total + MinutosProduccion_total`    
Entonces si calculo la diferencia entre Preparación y Producción debe dar 0


In [518]:
# ============================================================
# DEPURACIÓN: fases con tiempo inflado por MES sin cerrar
# ============================================================
# Criterio: un tiempo de preparación o producción desproporcionado
# (>= 480 min ≈ 1 turno) junto con un nº elevado de imputaciones
# indica que la máquina quedó abierta en el MES acumulando tiempo
# sin trabajo real.

UMBRAL_TIEMPO = 480          # minutos (1 turno de 8h) en prep o prod
UMBRAL_IMPUTACIONES = 3     # nº de imputaciones sospechoso

fase_agregada["mes_sin_cerrar"] = (
    (
        (fase_agregada["MinutosPreparacion_total"] >= UMBRAL_TIEMPO)
        | (fase_agregada["MinutosProduccion_total"] >= UMBRAL_TIEMPO)
    )
    & (fase_agregada["n_imputaciones"] > UMBRAL_IMPUTACIONES)
)

# reporte antes de eliminar
sospechosas = fase_agregada[fase_agregada["mes_sin_cerrar"]]
print(f"Fases marcadas como MES sin cerrar: {len(sospechosas)}")
print(sospechosas.nlargest(10, "MinutosProduccion_total")[
    ["SmFaseUid", "MaquinaDominante", "smFaseNme",
     "MinutosPreparacion_total", "MinutosProduccion_total", "n_imputaciones"]
].to_string(index=False))

antes = len(fase_agregada)
fase_agregada = fase_agregada[~fase_agregada["mes_sin_cerrar"]]
print(f"\nFases antes: {antes} | eliminadas: {antes - len(fase_agregada)} | después: {len(fase_agregada)}")

fase_agregada = fase_agregada.drop(columns=["mes_sin_cerrar"])

Fases marcadas como MES sin cerrar: 1159
                           SmFaseUid MaquinaDominante smFaseNme  MinutosPreparacion_total  MinutosProduccion_total  n_imputaciones
5D715E06-67F3-451C-AE5E-982E0083402F             CM04    CV-ALU                4968.85000              32412.53333              77
EDD89008-F84F-427B-802D-24BFEA25DEE4             CM17     MEC03                 585.76666              30921.84999              74
B1378B1E-8601-4B4F-A08C-501D8A0703E6             CM17     MEC03                 585.29999              30917.48332              74
BDD4BE9B-9BCF-4BCF-BCF4-E1BBB9E780DC             CM17     MEC03                 585.71666              30917.48332              74
1A5CBCD3-3A66-4C09-82A2-ADF8A9BE0E3A             CM17     MEC03                2243.95000              28929.40000              74
3A56FC28-C0C7-4219-9701-C91B548BB507             CM17     MEC03                2243.95000              28929.40000              74
68416ED9-521E-405A-8DB1-0D42D2CDDF84      

In [519]:
display(fase_agregada.sample(3))

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,n_imputaciones
4714,41B9C845-07DA-45C5-82E3-574848FE1B2F,TP01,OF2505445,PE2XNCX0112,MTP,Allonge Coudée 103104221,10.35000,10.35000,0.0,2
11006,98E1755B-9676-45DE-B762-E8C7DCF0AD5D,TP02,OF2512538,PE1XNCX0218,MTP,Elektrodenaufnahme F704501052840,62.33333,62.33333,0.0,2
12212,A8605027-477D-48C9-9991-0186AD8FF8AD,TN09,OF2506216,PE1XA2X0419,DES,ALLONGE COUDÉE 062ZF20024431ZR1,25.61667,25.61667,0.0,2


> Hay imputaciones con un ruido inmanejable, voy a tener que eliminarlas.

Durante la validación de los tiempos registrados en el sistema de captura de planta (MES) se observó, en un subconjunto de fases, una incoherencia entre el tiempo real imputado y la suma de sus componentes de preparación y producción. Al analizar estos casos, se identificó que las desviaciones más severas se concentraban en fases con un número anormalmente elevado de imputaciones. La inspección manual de estas fases reveló el origen del problema: en determinadas ocasiones, una máquina permanecía con la orden abierta en el MES tras finalizar el trabajo efectivo, de modo que el sistema continuaba acumulando imputaciones de tiempo real de forma automática, sin actividad productiva real asociada. Este tiempo espurio inflaba de forma muy acusada el tiempo real de la fase.

Dado que el tiempo real constituye el numerador de la variable objetivo del estudio (el ratio entre tiempo real y teórico), estos registros habrían distorsionado gravemente el cálculo de la idoneidad, atribuyendo desviaciones elevadas a asignaciones que en realidad fueron correctas y cuyo exceso de tiempo se debía únicamente a un fallo en el cierre del registro. Por este motivo, y tras verificar individualmente su naturaleza, se han descartado las fases afectadas por este fenómeno. El criterio de identificación combinó la magnitud de la desviación con el número de imputaciones de la fase, lo que permitió aislar los registros defectuosos sin eliminar aquellas fases cuya ligera desviación responde a las condiciones normales de imputación del trabajo. Tras esta depuración, el conjunto de análisis queda constituido por 17.326 fases con tiempos fiables.

In [520]:
# Voy añadir el grupo de la máquina a la tabla final de imputaciones agregadas por fase
df_fases_MES = fase_agregada.merge(maquina_grupo, left_on="MaquinaDominante", right_on="CodigoMaquina", how="left")

In [521]:
# Verificar si alguna fase se quedo sin grupo máquina:
print(f"Fases sin grupo tras el merge: {df_fases_MES['NombreGrupo'].isna().sum()}")

Fases sin grupo tras el merge: 0


In [ ]:
df_fases_MES = df_fases_MES.drop("CodigoMaquina", axis=1)

display(df_fases_MES.sample(1))
display(f"Grupos de máquinas: {len(df_fases_MES["CodigoGrupo"].unique())}")

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,n_imputaciones,IDMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
4828,479235DB-E5BF-4229-A273-D10C69543C1F,TN09,OF2506865,PE3JNCX0004,TPP,ALLONGE COUDÉE 103150447,46.31667,46.31667,0.0,2,0bab3b63-5d31-4de3-85e2-89dc5a3e9bd5,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,27451f15-090c-42a6-82d6-98b7451db3db,TPU,Torns Puma (Taller COURES)


'Grupos de máquinas: 9'

In [524]:
df_fases_MES.to_parquet("res//procesed/df_fases_MES.parquet")